# AC Performance

**BQ2:** How can seasonal weather changes, air-con performance and humidity levels be analysed to detect zones that are at risk of air-con degradation or mould growth?

Compares July-August 2018 with 2019 for Floors 3-7, zones 1, 2, 4 and 5, with and without filled readings. It produces `outputs/bq2-zone-screen/inspection_shortlist.csv`, which the risk matrix (`bq2_risk_matrix.ipynb`) uses for the **AC energy up** and **faulty AC meter** labels.

## Methods

| Step | Rule |
|---|---|
| Inputs | Existing raw/cleaned minutes and cached weather; no new filling or partial-day scaling. Observed-only readings must be present and unchanged by cleaning. |
| Energy | Inferred working days with all 1,440 AC-plus-plug minutes; every assigned AC meter required. |
| Environment | Continuous 60-minute startup windows; complete steady-on hours after 120 AC-on minutes. Thresholds are frozen from raw-observed 2018. |
| Reference | Fit/select using 2018 only: temperature-plus-plug versus enthalpy-plus-plug, expanding temporal validation and mean/activity benchmarks. |
| Comparability | At least 20 reference days/four weeks; at least 10 evaluation days/four weeks inside the 2018 predictor convex hull, by session where relevant. |
| Uncertainty | Equal total day weights; 2,000 calendar-week bootstrap refits with fixed specification/cohort. Nominal conditional 95% intervals, without multiplicity adjustment. |
| Interpretation | Stronger leads require reference MAE ≤ mean-only MAE, ≥90% valid bootstrap replicates and an adverse interval in both input versions. Weak references remain labelled; mean-only sensitivity uses the same supported rows. |

These are project screening rules. Working-day/plug labels are activity proxies; 2018 is not known healthy. Missing results do not establish healthy equipment. Clock alignment, calibration, controls and loads remain uncertain. Full tables are saved as CSVs; intermediate displays are previews.

In [5]:
pip install holidays

  Obtaining dependency information for holidays from https://files.pythonhosted.org/packages/d7/af/cc7abdb463668cf581780fff8636254892139f81fad95977e594cb217bfb/holidays-0.108-py3-none-any.whl.metadata
     ---------------------------------------- 0.0/57.2 kB ? eta -:--:--
     --------------------------- ---------- 41.0/57.2 kB 960.0 kB/s eta 0:00:01
     -------------------------------------- 57.2/57.2 kB 594.0 kB/s eta 0:00:00
  Obtaining dependency information for python-dateutil<3,>=2.9.0.post0 from https://files.pythonhosted.org/packages/ec/57/56b9bcc3c9c6a792fcbaf139543cee77261f3651ca9da0c93f5c1221264b/python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
    --------------------------------------- 0.0/1.7 MB ? eta -:--:--
   - -------------------------------------- 0.1/1.7 MB 1.1 MB/s eta 0:00:02
   - -------------------------------------- 0.1/1.7 MB 653.6 kB/s eta 0:00:03
   -- ----------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
conda-repo-cli 1.0.41 requires requests_mock, which is not installed.
conda-repo-cli 1.0.41 requires clyent==1.2.1, but you have clyent 1.2.2 which is incompatible.
conda-repo-cli 1.0.41 requires nbformat==5.4.0, but you have nbformat 5.7.0 which is incompatible.
conda-repo-cli 1.0.41 requires python-dateutil==2.8.2, but you have python-dateutil 2.9.0.post0 which is incompatible.
conda-repo-cli 1.0.41 requires requests==2.28.1, but you have requests 2.31.0 which is incompatible.


In [6]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

# Project root = the folder holding data-cleaning/; this BQ2 folder sits in analytics/ inside it
here = Path.cwd().resolve()
repo_root = next(p for p in (here, *here.parents) if (p / "data-cleaning").is_dir())
analysis_dir = repo_root / "analytics/mould-ac-degradation"
sys.path.insert(0, str(analysis_dir / "src"))
helper_notebook = analysis_dir / "bq2_utils.ipynb"
get_ipython().run_line_magic("run", f'-i "{helper_notebook}"')

from bq2 import zone_screen as screen
sys.path.insert(0, str(repo_root / BASELINE_CODE_DIR))
from humidity_risk_baseline import load_bangkok_era5_weather
output_dir = Path(analysis_dir / "outputs/bq2-zone-screen")
output_dir.mkdir(parents=True, exist_ok=True)
weather = load_bangkok_era5_weather(repo_root / WEATHER_CSV)
helpers = {name: globals()[name] for name in ["add_outdoor_psychrometrics", "humidity_ratio_gkg", "ac_on_threshold"]}
pd.set_option("display.max_columns", 14)
pd.set_option("display.max_rows", 8)

## 1. Check inputs

Audit timestamps, channel validity, coverage, filling and hourly reconciliation before fitting. Floor 6's empty malformed tail row is excluded locally; trailing minutes remain missing. Floor 7 uses its explicit day-first source format.

In [11]:
%%capture --no-display --no-stderr
evidence = screen.audit_and_features(repo_root, weather, helpers, output_dir)
quality = evidence["zone_quality"]
display(quality[["zone_id", "ac_meters", "full_clean_joint_coverage_pct",
                 "with_fills_2018_joint_coverage_pct", "with_fills_2019_joint_coverage_pct",
                 "suspect_sensor", "meter_mapping_review"]].round(2))
display(evidence["timestamp_audit"].query("invalid_timestamps > 0"))
assert evidence["channel_quality"].quality_log_missing_agrees.all()
assert evidence["hourly_reconciliation"][["availability_mismatches", "value_mismatches"]].sum().sum() == 0
print("All 95 relevant channel missingness records and 80 hourly comparisons reconcile.")
print("Longest requested-window raw plateaus:")
display(evidence["sensor_checks"].groupby("stream").raw_scope_max_constant_minutes.max())


,zone_id,ac_meters,full_clean_joint_coverage_pct,with_fills_2018_joint_coverage_pct,with_fills_2019_joint_coverage_pct,suspect_sensor,meter_mapping_review
0,F3-Z1,4,61.92,99.20,98.66,False,False
1,F3-Z2,1,64.88,99.07,98.92,False,False
2,F3-Z4,1,64.84,97.49,98.59,False,False
3,F3-Z5,1,54.80,80.97,96.78,False,False
...,...,...,...,...,...,...,...
16,F7-Z1,4,66.75,98.28,98.14,False,True
17,F7-Z2,1,67.46,98.67,98.29,False,False
18,F7-Z4,1,66.58,99.08,98.64,False,False
19,F7-Z5,1,64.91,98.13,98.85,False,False


,source,rows,invalid_timestamps,date_format,invalid_rows_with_measurements,duplicate_timestamps,sorted,first,last,off_minute_grid
13,data\2019Floor6.csv,424169,1,%Y-%m-%d %H:%M:%S,0,0,True,2019-01-01 00:00:00,2019-10-22 13:27:00,0


stream
RH             43
paired         34
temperature    71
Name: raw_scope_max_constant_minutes, dtype: int64

F4-Z1 and F7-Z1 have near-zero AC channels requiring meter/configuration checks. F5-Z1 environmental readings are held for sensor verification; energy is assessed separately. These flags do not diagnose failed equipment or verify sensor placement.

## 2. Fit eligible comparisons

Keep unavailable metrics and their reasons. Energy may remain usable during environmental outages. Reference models that miss the simple benchmark retain a validation caution.

In [12]:
%%capture --no-display --no-stderr
result = screen.run_screen(evidence, output_dir, repetitions=2000)
summary = result["model_summary"]
display(summary.groupby(["analysis", "variant"]).agg(
    fitted=("reference_cv_MAE", "count"), supported=("reportable", "sum"),
    passes_reference_validation=("lead_eligible", "sum")))
cv = result["cross_validation"]
assert cv.date.dt.year.eq(2018).all()
assert pd.to_datetime(cv.train_last_date).lt(cv.date).all()
assert result["shared_cohort_checks"].max_abs_input_or_target_difference.dropna().lt(1e-8).all()
assert not summary.query("analysis == 'cooling'").reportable.any()
print("No cooling comparison meets the coverage/common-support rules.")


fitted  supported  passes_reference_validation
analysis variant                                                      
cooling  with_fills         13          0                            0
         without_fills      12          0                            0
energy   with_fills         18         14                            7
         without_fills      13         11                            6
moisture with_fills         11          8                            3
         without_fills      10          8                            1

## 3. Compare references and inspection leads

The mean-only sensitivity uses each model's exact supported cohort. It tests dependence on the weather/session reference, not a weather-adjusted efficiency change. The saved `inspection_shortlist.csv` lists every zone and its status.

In [13]:
benchmark = screen.benchmark_sensitivity(evidence, result, output_dir, repetitions=2000)
shortlist = screen.inspection_shortlist(quality, summary, benchmark)
shortlist.to_csv(output_dir / "inspection_shortlist.csv", index=False)
display(shortlist[["zone_id", "inspection_category", "validated_metrics_in_both_versions", "validation_passes_only_one_version"]])
focus = ["F3-Z4", "F5-Z4", "F7-Z2"]
display(summary[summary.zone_id.isin(focus) & summary.analysis.isin(["energy", "moisture"])][[
    "zone_id", "analysis", "variant", "supported_days", "mean_residual", "residual_pct",
    "bootstrap_low", "bootstrap_high", "reference_cv_MAE", "mean_only_cv_MAE", "lead_eligible"]].round(3))
display(benchmark[benchmark.zone_id.isin(focus)].round(3))


,zone_id,inspection_category,validated_metrics_in_both_versions,validation_passes_only_one_version
0,F3-Z1,Review operating change,1,
1,F3-Z2,Review operating change,1,
2,F3-Z4,Inspect energy and controls,2,
3,F3-Z5,Insufficient comparable evidence,0,
...,...,...,...,...
16,F7-Z1,Verify AC meter mapping,0,
17,F7-Z2,Inspect environment; validate reference,1,moisture
18,F7-Z4,Insufficient comparable evidence,0,
19,F7-Z5,No consistent adverse departure,1,


,zone_id,analysis,variant,supported_days,mean_residual,residual_pct,bootstrap_low,bootstrap_high,reference_cv_MAE,mean_only_cv_MAE,lead_eligible
12,F3-Z4,energy,with_fills,22,17.493,13.275,8.103,25.789,14.098,16.745,True
13,F3-Z4,energy,without_fills,10,19.943,15.207,7.120,30.326,14.098,16.745,True
16,F3-Z4,moisture,with_fills,23,-0.929,-9.661,-1.065,-0.802,0.241,0.245,True
17,F3-Z4,moisture,without_fills,19,-0.929,-9.669,-1.051,-0.783,0.186,0.245,True
...,...,...,...,...,...,...,...,...,...,...,...
102,F7-Z2,energy,with_fills,20,-11.261,-5.164,-23.954,1.464,13.385,14.660,True
103,F7-Z2,energy,without_fills,14,-11.486,-5.289,-24.423,0.462,13.730,15.505,True
106,F7-Z2,moisture,with_fills,33,1.151,11.775,0.917,1.412,0.268,0.292,True
107,F7-Z2,moisture,without_fills,29,1.137,11.617,0.926,1.337,0.284,0.272,False


,zone_id,analysis,variant,supported_days,mean_only_residual,mean_only_residual_pct,bootstrap_low,bootstrap_high,bootstrap_percent_low,bootstrap_percent_high
8,F3-Z4,energy,with_fills,22,17.047,12.893,7.641,27.807,5.445,22.280
9,F3-Z4,energy,without_fills,10,18.871,14.272,7.897,31.552,5.646,25.602
10,F3-Z4,moisture,with_fills,23,-0.912,-9.501,-1.048,-0.785,-10.816,-8.273
11,F3-Z4,moisture,without_fills,19,-0.893,-9.328,-1.028,-0.768,-10.633,-8.122
...,...,...,...,...,...,...,...,...,...,...
35,F7-Z2,energy,with_fills,20,-10.980,-5.041,-24.561,2.208,-10.772,1.037
36,F7-Z2,energy,without_fills,14,-12.665,-5.801,-26.700,1.162,-11.629,0.561
37,F7-Z2,moisture,with_fills,33,1.175,12.051,0.921,1.416,9.445,14.568
38,F7-Z2,moisture,without_fills,29,1.158,11.858,0.920,1.379,9.376,14.161


### Findings

- **F3-Z4:** energy departure +13.3%/+15.2% with/without fills, with a reference that improves on the mean benchmark. Inspect operating conditions; lower recorded moisture and unavailable cooling comparisons do not establish degradation.
- **F5-Z4:** +36.7% in both versions, but reference validation is weak. The same-cohort mean sensitivity is +33.0%, supporting an electrical/operating investigation.
- **F7-Z2:** recorded moisture +1.15/+1.14 g/kg on 33/29 days, with an observed-only validation caution. Sensor, ventilation, moisture-load and control explanations remain open.
- F3-Z1/F3-Z2 show lower supported energy. Other zones retain QA, provisional or unavailable statuses; none is certified healthy.